In [6]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from xgboost import XGBClassifier

import torch
from torch.utils.data import TensorDataset, DataLoader


from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

# Split arrays or matrices into random train and test subsets
from sklearn.model_selection import train_test_split

# Encode target labels with value between 0 and n_classes-1, Standardize features by removing the mean and scaling to unit variance.
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Logistic Regression (aka logit, MaxEnt) classifier
from sklearn.linear_model import LogisticRegression

# Accuracy classification score, Build a text report showing the main classification metrics, Compute confusion matrix to evaluate the accuracy of a classification
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [7]:
data_path = (
    project_root
    / "data"
    / "processed"
    / "pbmc3k_clustered_annotated.h5ad"
)

adata = sc.read_h5ad(data_path)

print(adata)
print(adata.obs["cell_type"].value_counts())

AnnData object with n_obs × n_vars = 2633 × 2000
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'leiden', 'cell_type'
    var: 'gene_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'
    layers: 'counts', None (.X)
cell_type
CD4 T cells        1175
CD14 Monocytes      639
NK cells            425
B cells             342
Dendritic cells      43
Platelets             9
Name: count, dtype: int64


In [8]:
# use normalized/log-transformed expression from adata.raw
X = adata.raw.X
y = adata.obs["cell_type"].astype(str).to_numpy()

# Convert sparse matrix
if sparse.issparse(x):
    X = X.toarray()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nClass counts:")
print(adata.obs["cell_type"].value_counts())

NameError: name 'x' is not defined

In [9]:
hvg_mask  = adata.raw.var_names.isin(
    adata.var_names[adata.var["highly_variable"]]
)

X_hvg = adata.raw.X[:, hvg_mask]

if sparse.issparse(X_hvg):
    X_hvg = X_hvg.toarray()

feature_names = adata.raw.var_names[hvg_mask].to_numpy()

print("X_hvg shape:", X_hvg.shape)
print("Number of features:", len(feature_names))
print("First 10 genes:", feature_names[:10])

X_hvg shape: (2633, 2000)
Number of features: 2000
First 10 genes: ['ISG15' 'TNFRSF4' 'CPSF3L' 'ATAD3C' 'C1orf86' 'RER1' 'TNFRSF25' 'TNFRSF9'
 'CA6' 'CTNNBIP1']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_hvg,
    y,
    test_size = 0.20,
    random_state = 42,
    stratify = y
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTraining class counts:")
print(pd.Series(y_train).value_counts())

print("\nTest class counts:")
print(pd.Series(y_test).value_counts())

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
log_reg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

log_reg.fit(X_train_scaled, y_train)

In [ ]:
y_pred = log_reg.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)

print("Test accuracy:", accuracy)
print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=3))

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=log_reg.classes_
)

print("Classes:")
print(log_reg.classes_)

print("\nConfusion matrix:")
print(cm)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=log_reg.classes_,
    xticks_rotation=45,
    cmap="Blues"
)

plt.title("Logistic Regression — PBMC3K Test Set")
plt.tight_layout()
plt.show()

In [ ]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Cell type mapping:")

for number, cell_type in enumerate(label_encoder.classes_):
    print(f"{number}: {cell_type}")

In [ ]:
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss"
)

In [ ]:
xgb_model.fit(
    X_train,
    y_train_encoded
)

In [ ]:
xgb_pred_encoded = xgb_model.predict(X_test)

xgb_pred = label_encoder.inverse_transform(
    xgb_pred_encoded
)

xgb_accuracy = accuracy_score(
    y_test,
    xgb_pred
)

print("XGBoost Test Accuracy:", xgb_accuracy)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        xgb_pred,
        digits=3
    )
)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    xgb_pred,
    labels=label_encoder.classes_,
    xticks_rotation=45,
    cmap="Blues"
)

plt.title("XGBoost - PBMC3K Test Set")
plt.tight_layout()
plt.show()

In [ ]:
model_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "XGBoost"
    ],
    "Accuracy": [
        0.9848197343453511,
        0.9791271347248577
    ],
    "Macro F1": [
        0.989,
        0.921
    ],
    "Weighted F1": [
        0.985,
        0.979
    ]
})

model_results

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators = 300,
    class_weight = "balanced",
    random_state = 42,
    n_jobs = -1
)

rf_model.fit(X_train, y_train)

In [ ]:
rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

print("Random Forest Test Accuracy:", rf_accuracy)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        rf_pred,
        digits=3
    )
)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    rf_pred,
    labels=rf_model.classes_,
    xticks_rotation=45,
    cmap="Blues"
)

plt.title("Random Forest - PBMC3K Test Set")
plt.tight_layout
plt.show()

In [ ]:
from sklearn.metrics import f1_score

model_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "XGBoost",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, xgb_pred),
        accuracy_score(y_test, rf_pred)
    ],
    "Macro F1": [
        f1_score(y_test, y_pred, average="macro"),
        f1_score(y_test, xgb_pred, average="macro"),
        f1_score(y_test, rf_pred, average="macro")
    ],
    "Weighted F1": [
        f1_score(y_test, y_pred, average="weighted"),
        f1_score(y_test, xgb_pred, average="weighted"),
        f1_score(y_test, rf_pred, average="weighted")
    ]
})

model_results.sort_values(
    "Macro F1",
    ascending=False
)

In [ ]:
log_reg_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
cv_results = cross_validate(
    log_reg_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "macro_f1": "f1_macro",
        "weighted_f1": "f1_weighted"
    },
    n_jobs=-1
)

print("Accuracy:", cv_results["test_accuracy"])
print("Macro F1:", cv_results["test_macro_f1"])
print("Weighted F1:", cv_results["test_weighted_f1"])

print("\nMean Accuracy:", cv_results["test_accuracy"].mean())
print("Mean Macro F1:", cv_results["test_macro_f1"].mean())
print("Mean Weighted F1:", cv_results["test_weighted_f1"].mean())

In [ ]:
xgb_cv_results = cross_validate(
    xgb_model,
    X_train,
    y_train_encoded,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "macro_f1": "f1_macro",
        "weighted_f1": "f1_weighted"
    },
    n_jobs=-1
)

print("Accuracy:", xgb_cv_results["test_accuracy"])
print("Macro F1:", xgb_cv_results["test_macro_f1"])
print("Weighted F1:", xgb_cv_results["test_weighted_f1"])

print("\nMean Accuracy:", xgb_cv_results["test_accuracy"].mean())
print("Mean Macro F1:", xgb_cv_results["test_macro_f1"].mean())
print("Mean Weighted F1:", xgb_cv_results["test_weighted_f1"].mean())

In [ ]:
rf_cv_results = cross_validate(
    rf_model,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "macro_f1": "f1_macro",
        "weighted_f1": "f1_weighted"
    },
    n_jobs=-1
)

print("Accuracy:", rf_cv_results["test_accuracy"])
print("Macro F1:", rf_cv_results["test_macro_f1"])
print("Weighted F1:", rf_cv_results["test_weighted_f1"])

print("\nMean Accuracy:", rf_cv_results["test_accuracy"].mean())
print("Mean Macro F1:", rf_cv_results["test_macro_f1"].mean())
print("Mean Weighted F1:", rf_cv_results["test_weighted_f1"].mean())

In [ ]:
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    estimator=log_reg_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

In [ ]:
print("Best C:", grid_search.best_params_)
print("Best CV Macro F1:", grid_search.best_score_)

In [ ]:
cv_tuning_results = pd.DataFrame(
    grid_search.cv_results_
)[[
    "param_classifier__C",
    "mean_train_score",
    "mean_test_score",
    "std_test_score"
]]

cv_tuning_results

In [ ]:
best_log_reg = grid_search.best_estimator_

In [ ]:
best_y_pred = best_log_reg.predict(X_test)

print(
    classification_report(
        y_test,
        best_y_pred,
        digits=3
    )
)

print(
    "Final Test Accuracy:",
    accuracy_score(y_test, best_y_pred)
)

print(
    "Final Macro F1:",
    f1_score(y_test, best_y_pred, average="macro")
)

print(
    "Final Weighted F1:",
    f1_score(y_test, best_y_pred, average="weighted")
)

In [ ]:
X_nn_train, X_val, y_nn_train, y_val = train_test_split(
    X_train_scaled,
    y_train_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_train_encoded
)

print("NN training shape:", X_nn_train.shape)
print("Validation shape:", X_val.shape)

In [ ]:

X_nn_train_tensor = torch.tensor(
    X_nn_train,
    dtype=torch.float32
)

y_nn_train_tensor = torch.tensor(
    y_nn_train,
    dtype=torch.long
)

X_val_tensor = torch.tensor(
    X_val,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.long
)

train_dataset = TensorDataset(
    X_nn_train_tensor,
    y_nn_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))